# Notebook 03

Objetivo:
Esta análise investiga como a classificação de gêneros e o nível de hibridismo (mistura de categorias) de um livro impactam a sua descoberta por novos leitores e a sua avaliação final. O objetivo prático é extrair regras lógicas para o motor de recomendação e gamificação do **Booklog**.

> ### ⚠️ Delimitação do Estudo
> Esta análise baseia-se **exclusivamente no conjunto de dados retirados do Kaggle**. Os padrões e associações mapeados refletem o comportamento histórico desta base específica e não devem ser generalizados como uma regra universal para todo o mercado editorial global.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import seaborn as sns
from itertools import combinations
from collections import Counter
import plotly.express as px
from itertools import combinations
import plotly.graph_objects as go
from plotly.subplots import make_subplots


sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.autolayout': True})
df = pd.read_parquet('datasets/books.parquet')
df = df.dropna(subset=['isbn'])
df = df.drop_duplicates(subset=['isbn'])
num_cols = ['pages', 'rating', 'reviews', 'totalratings']
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.dropna(subset=['rating', 'totalratings', 'genre'])
df['genre_list'] = df['genre'].apply(lambda x: list(set(x)) if isinstance(x, list) else [])

## 1. Engenharia de Variáveis (Feature Engineering)

Esta etapa transforma os dados brutos em métricas analíticas e limpa a taxonomia para os modelos seguintes.
* **Taxa de Engajamento (`engagement_ratio`):** Cruza a coluna `reviews` com `totalratings`. Ela mede a proporção de leitores que se deram ao trabalho de *escrever uma resenha textual* em relação aos que apenas clicaram nas estrelas.
* **Rating Bayesiano:** Ajusta a nota ponderando-a pelo volume de avaliações, eliminando o viés de livros com nota máxima baseada em pouquíssimos votos.
* **Filtro de Gêneros e Hibridismo:** Transforma o texto em listas, remove tags "guarda-chuva" (*Fiction*, *Books*, etc.) e calcula a quantidade (`genre_count`) de gêneros válidos de cada obra.

In [2]:
df['engagement_ratio'] = np.where(df['totalratings'] > 0, df['reviews'] / df['totalratings'], 0)
C = df['rating'].mean()
m = df['totalratings'].quantile(0.25)
df['bayesian_rating'] = (df['totalratings'] / (df['totalratings'] + m) * df['rating']) + (m / (df['totalratings'] + m) * C)

df['genre'] = df['genre'].fillna('')
df['genre_list'] = df['genre'].apply(lambda x: [g.strip() for g in str(x).split(',') if g.strip()])


stop_genres = {'Fiction', 'Literature', 'Books', 'Audiobook', 'Audiobooks', 'Nonfiction'}
df['genre_list'] = df['genre_list'].apply(lambda x: [g for g in x if g not in stop_genres])
df['genre_count'] = df['genre_list'].apply(len)

## 2. Redução de Dimensionalidade Estrutural (Princípio de Pareto)

Aplicamos o Princípio de Pareto como critério estatístico para a redução de dimensionalidade da base de dados, permitindo filtrar e isolar as tags de gêneros literários que possuem real relevância volumétrica para a plataforma.

* **Frequência e Distribuição Acumulada:** O algoritmo mapeia a frequência absoluta de cada gênero e calcula a sua porcentagem cumulativa em ordem decrescente, estabelecendo uma função de distribuição acumulada empírica.
* **Análise de Eixo Duplo Interativa:** Integra a contagem volumétrica (eixo Y primário) com a curva de percentual acumulado (eixo Y secundário), parametrizando visualmente o limiar crítico de corte.

**O Princípio de Pareto Aplicado ao Acervo:**
Ao ordenar as categorias por dominância, a modelagem estatística provou que **apenas 205 gêneros (representando 17,7% das 1.158 tags existentes no total) concentram exatamente 80% do volume de todo o acervo**. 

Os outros 82,3% dos gêneros restantes configuram o fenômeno da **Cauda Longa** (*Long Tail*): uma massa ruidosa de micro-nichos e redundâncias. Esse corte em 205 categorias serve como o funil exato que alimenta e valida o nosso motor de associação por Lift.

In [3]:
all_genres = [genre for sublist in df['genre_list'] for genre in sublist]
genre_counts = Counter(all_genres)

pareto_df = pd.DataFrame.from_dict(genre_counts, orient='index', columns=['Frequencia']).sort_values('Frequencia', ascending=False)
pareto_df['Porcentagem_Cumulativa'] = pareto_df['Frequencia'].cumsum() / pareto_df['Frequencia'].sum() * 100

plot_df = pareto_df[pareto_df['Porcentagem_Cumulativa'] <= 82].reset_index()
plot_df.rename(columns={'index': 'Genero'}, inplace=True)

fig = make_subplots(specs=[[{"secondary_y": True}]])


fig.add_trace(
    go.Bar(
        x=plot_df['Genero'],
        y=plot_df['Frequencia'],
        name="Frequência",
        marker_color='#2C3E50'
    ),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(
        x=plot_df['Genero'],
        y=plot_df['Porcentagem_Cumulativa'],
        name="% Cumulativa",
        mode='lines+markers',
        line=dict(color='#E74C3C', width=3),
        marker=dict(size=4)
    ),
    secondary_y=True,
)

fig.add_hline(
    y=80,
    line_dash='dash',
    line_color='#7F8C8D',
    line_width=2,
    annotation_text='Corte de Pareto (80%)',
    annotation_position='top left',
    annotation_font_color='#7F8C8D',
    secondary_y=True
)

fig.update_layout(
    title_text='<b>Gráfico de Pareto: Os Gêneros que Sustentam o Acervo</b>',
    title_font_size=16,
    template='plotly_white',
    hovermode='x unified',
    showlegend=False,
    autosize=True,
    height=600
)

fig.update_xaxes(title_text="Gêneros Literários", tickangle=45, nticks=40)
fig.update_yaxes(title_text="Frequência Absoluta (Contagem de Tags)", secondary_y=False)
fig.update_yaxes(title_text="Porcentagem Cumulativa (%)", range=[0, 105], secondary_y=True)

fig.show()

## 3. Modelagem de Associação (Lift) e Limites Metodológicos

Aplicamos a métrica **Lift** para quantificar a força de co-ocorrência entre gêneros, isolando previamente as 205 categorias mais frequentes (Pareto).

**Procedimento:**
- **Cálculo probabilístico:** `lift = P(A∩B) / (P(A) * P(B))`
- **Filtros heurísticos:** Remoção de redundâncias via (i) dicionário manual de sinônimos, (ii) interseção lexical de palavras com `len(w) > 3`, e (iii) stemming simples de sufixos comuns (`-tion`, `-ing`, `-ical`, etc.). Pares com frequência `< 50` foram descartados.

**Limitação:** A análise restringe-se à co-ocorrência nominal dos rótulos da folksonomia. O tratamento de redundância é determinístico e manual, sem inferência semântica profunda (NLP). Solução leve e eficaz para *cold start*, porém limitada à superfície da tagagem comunitária.

In [4]:
df['genre_list'] = df['genre_list'].apply(lambda x: list(set(x)))
N = len(df)
all_genres = [genre for sublist in df['genre_list'] for genre in sublist]
genre_counts = Counter(all_genres)
genre_probs = {genre: count / N for genre, count in genre_counts.items()}
top_205_genres = set([genre for genre, count in genre_counts.most_common(167)])

# Função para detectar redundância semântica e lexical
def is_redundant(g1, g2):

    def stem_simple(word):
        suffixes = ['tion', 'sion', 'ing', 'ian', 'ical', 'ous', 'ism', 'ist', 'ary', 'ery', 'ory', 'al', 'ic', 'an', 'en', 'er', 'ed', 'es', 's']
        for s in sorted(suffixes, key=len, reverse=True):
            if word.endswith(s) and len(word) - len(s) >= 4:
                return word[:-len(s)]
        return word

    g1_clean = g1.lower().strip()
    g2_clean = g2.lower().strip()

    synonyms = [
            # Gêneros de Identidade
            {'glbt', 'lgbt', 'queer', 'gay', 'lesbian', 'yaoi', 'm m romance'},
            
            # Formatos Gráficos
            {'comics', 'graphic novels', 'graphic novels comics', 'sequential art', 'manga', 'comic book'},
            
            # Faixa Etária
            {'teen', 'young adult', 'juvenile', 'middle grade'},
            
            # Gastronomia
            {'cookbooks', 'cooking', 'food', 'food and drink'},
            
            # Redundâncias clássicas de Tensão
            {'thriller', 'suspense'},
            {'crime', 'detective'},
            
            # Autoajuda e Carreira
            {'self help', 'personal development', 'leadership'},
            
            # Ficção Feminina Clássica
            {'chick lit', 'womens fiction', 'womens'},
            
            # Literatura Biográfica
            {'memoir', 'autobiography', 'biography'},
            
            # Conteúdo Adulto
            {'erotica', 'menage'},
            
            # Tecnologia
            {'programming', 'technology', 'computer science', 'computers', 'software', 'tech'},
            
            # Ciências da Natureza
            {'science', 'popular science', 'science and technology', 'natural science'},
            
            # Dystopia e Pós-Apocalíptico
            {'dystopia', 'dystopian', 'post apocalyptic', 'apocalyptic'},
            
            # Política e Atualidades
            {'politics', 'political science', 'current events', 'current affairs'},
            
            # Saúde Mental
            {'psychology', 'psychiatry', 'mental health'},
            
            # Formato de Contos
            {'short stories', 'short story collections', 'anthologies', 'anthology'},
            
            # Magia e Ocultismo
            {'magic', 'witches', 'wizards', 'occult', 'supernatural', 'paranormal'},
            
            # Feriados
            {'christmas', 'holiday'},
            
            # Religião
            {'christian living', 'faith', 'christianity', 'christian', 'church', 'theology', 'christian fiction', 'christian romance', 'spirituality'},
            
            # História Militar
            {'civil war', 'military history'},
            
            # Artes Cênicas
            {'drama', 'plays'},
            
            # Ecologia
            {'environment', 'nature'},
            
            # Redação e Linguística
            {'essays', 'writing', 'language'},
            
            # Cultura Asiática
            {'asia', 'asian literature', 'japan'},
            
            # Ambiente Escolar
            {'academic', 'school'}
        ]

    pair_set = {g1_clean, g2_clean}
    for syn_group in synonyms:
        if pair_set.issubset(syn_group):
            return True

    # filtro lexical (palavras idênticas)
    words1 = set(g1_clean.replace('-', ' ').split())
    words2 = set(g2_clean.replace('-', ' ').split())
    shared_words = {w for w in words1.intersection(words2) if len(w) > 3}
    if shared_words:
        return True

    # filtro de radical (pega dystopia/dystopian, politic/political, etc.)
    stems1 = {stem_simple(w) for w in words1 if len(w) > 4}
    stems2 = {stem_simple(w) for w in words2 if len(w) > 4}
    if stems1.intersection(stems2):
        return True

    return False

# Probabilidade conjunta P(A ∩ B)
pair_counts = Counter()
df['genre_list'].apply(lambda x: pair_counts.update(list(combinations(sorted(x), 2))))

# Calcular o Lift com os Filtros Aplicados
edges_data = []
for (g1, g2), freq in pair_counts.items():

    if g1 not in top_205_genres or g2 not in top_205_genres:
        continue

    if is_redundant(g1, g2):
        continue

    if freq < 50:
        continue

    p_a_and_b = freq / N
    p_a = genre_probs[g1]
    p_b = genre_probs[g2]

    lift = p_a_and_b / (p_a * p_b)
    edges_data.append([g1, g2, freq, lift])

# Resultados processados
edge_df = pd.DataFrame(edges_data, columns=['Source', 'Target', 'Frequency', 'Lift'])
edge_df = edge_df.sort_values(by='Lift', ascending=False)

## 4. Visualização das Associações (Afinidade de Nichos)
### O que é o Lift e como ele funciona?
O **Lift** é uma métrica que mede a força de atração entre dois gêneros:
* **Lift = 1.0 (Limiar do Acaso):** Significa que os dois gêneros são totalmente independentes. Encontrá-los juntos em um livro é pura coincidência.
* **Lift maior que 1.0 (Conexão Real):** Significa que os gêneros têm afinidade.

**Linha Vermelha de Referência (Lift = 1.0):** Marca o limite do acaso. Tudo o que ultrapassa essa linha indica uma associação legítima.

In [5]:
top_lifts = edge_df.head(20).copy()
top_lifts['Pair_Name'] = top_lifts['Source'] + ' + ' + top_lifts['Target']

fig = px.bar(
    top_lifts,
    x='Lift',
    y='Pair_Name',
    orientation='h',
    title='<b>Top 20 Associações de Gêneros Literários</b>',
    color_discrete_sequence=['#2C3E50'],
    labels={'Lift': 'Força de Associação (Lift)', 'Pair_Name': 'Pares de Gêneros'}
)

fig.add_vline(
    x=0.0, 
    line_dash='dash', 
    line_color='#E74C3C', 
    line_width=2,
    annotation_text='Limiar de Independência (Lift = 1.0)', 
    annotation_position='bottom right',
    annotation_yshift=-40,
    annotation_font_color='#E74C3C'
)

fig.update_layout(
    title_font_size=16,
    xaxis_title='Força de Associação (Lift)',
    yaxis_title='Pares de Gêneros',
    yaxis={'categoryorder': 'total ascending'},
    template='plotly_white',
    autosize=True,
    height=600
)

fig.show()

## 5. Análise de Variância: O Impacto do Hibridismo (Boxplot)

Nesta etapa final, cruzamos a quantidade de tags atribuídas a uma obra com a sua Nota Bayesiana para investigar como o acúmulo de categorias afeta a satisfação do público.

**Filtro de Escopo:** Restringe a análise a livros que possuem de 1 a 5 gêneros, removendo obras com marcações excessivas.


In [6]:
import plotly.express as px

df_filtered = df[(df['genre_count'] > 0) & (df['genre_count'] <= 5)]

fig = px.box(
    df_filtered,
    x='genre_count',
    y='bayesian_rating',
    title='<b>Distribuição da Nota Bayesiana pelo Número de Gêneros Atribuídos</b>',
    labels={
        'genre_count': 'Quantidade de Gêneros no Mesmo Livro', 
        'bayesian_rating': 'Nota Corrigida (Bayesian Rating)'
    }
)

fig.update_traces(
    boxpoints=False, 
    fillcolor='#BDC3C7',
    line_color='#2C3E50',
    line_width=2
)

fig.update_layout(
    title_font_size=16,
    xaxis_title='Quantidade de Gêneros no Mesmo Livro',
    yaxis_title='Nota Corrigida (Bayesian Rating)',
    xaxis=dict(tickmode='linear', dtick=1),
    yaxis=dict(range=[0, 5]),
    template='plotly_white',
    autosize=True,
    height=600
)

fig.show()

## 6. Conclusão: Respondendo à Pergunta de Pesquisa
**1. A Eficiência da Cauda Longa (Pareto)**
Os dados provam que **apenas 17,7% das tags (205 de 1.158)** concentram 80% de todo o comportamento de leitura. Para o banco de dados da aplicação, isso justifica a criação de um índice de busca focado exclusivamente neste "grupo de elite", garantindo alta performance no servidor e eliminando o ruído gerado pela folksonomia esparsa.

**2. A Engenharia da Descoberta Orgânica (Motor de Lift)**
A retenção de leitores se apoia em mapear afinidades latentes fortes. O motor de Lift revelou conexões extraordinárias que vão muito além do acaso. Por exemplo, a probabilidade de um leitor engajar com *Paranormal Romance* ao ler *Vampires* é **22,7 vezes maior** do que a média. Essas métricas fornecem as regras preditivas para cruzar nichos no feed e resolver o problema de *cold start* para novos usuários de forma inteligente.

**3. O Limite do Hibridismo e a Gestão de Expectativas**
Os dados quebram a hipótese de que rotular uma obra com múltiplos gêneros aumenta o seu prestígio. A análise de variância demonstra exatamente o oposto:
* A nota mediana mantém-se estagnada em **3.9** (ou 3.91), independentemente de o livro possuir 1, 2 ou até 5 gêneros.
* A polarização das notas, no entanto, sofre um aumento sensível: o desvio padrão sobe de **0.14** (livros de 1 gênero) para **0.21** (livros de 5 gêneros).

### Veredito Final
O Booklog pode focar de forma cirúrgica nos 205 gêneros do Princípio de Pareto, aplicando as fortíssimas pontes probabilísticas do Lift para popular o motor de recomendação, e utilizando a contagem de gêneros de uma obra como um termômetro estatístico para calibrar a margem de risco(leve) das indicações feitas pelo algoritmo no feed principal.